# Downstream Dataset Evaluation

Verify all 9 processed EEG datasets: load, inspect, train a minimal model, report results.

In [ ]:
import sys, os, warnings
sys.path.insert(0, '..')
os.environ["SLURM_JOB_NAME"] = "bash"
warnings.filterwarnings("ignore", "Can't initialize NVML")
warnings.filterwarnings("ignore", category=FutureWarning)

import torch
import torch.nn as nn
import pytorch_lightning as pl
from pytorch_lightning.callbacks import EarlyStopping
from torch.utils.data import DataLoader, random_split, Subset
from speed.dataloader import DownstreamDataset, get_weighted_sampler

torch.set_float32_matmul_precision("medium")
pl.seed_everything(42, workers=True)

PROCESSED = "/scratch/agjma/SPEED/Processed"
MAX_EPOCHS = 30
MAX_SAMPLES = 5000  # cap for quick iteration


def z_normalize(x):
    """Per-channel zero-mean unit-variance normalization."""
    return (x - x.mean(dim=-1, keepdim=True)) / (x.std(dim=-1, keepdim=True) + 1e-8)


class SimpleEEG(pl.LightningModule):
    """Minimal 1D-CNN for EEG classification or regression."""

    def __init__(self, n_channels, n_outputs, task="classification"):
        super().__init__()
        self.save_hyperparameters()
        self.task = task
        self.conv = nn.Sequential(
            nn.Conv1d(n_channels, 32, 7, padding=3), nn.ReLU(), nn.BatchNorm1d(32),
            nn.Conv1d(32, 64, 7, padding=3), nn.ReLU(), nn.AdaptiveAvgPool1d(1),
        )
        self.head = nn.Linear(64, n_outputs)
        self.loss_fn = nn.CrossEntropyLoss() if task == "classification" else nn.MSELoss()

    def forward(self, x):
        return self.head(self.conv(x).squeeze(-1))

    def _step(self, batch, prefix):
        x, y = batch
        logits = self(x)
        loss = self.loss_fn(logits, y)
        self.log(f"{prefix}_loss", loss, on_epoch=True, on_step=False, prog_bar=True)
        if self.task == "classification":
            acc = (logits.argmax(1) == y).float().mean()
            self.log(f"{prefix}_acc", acc, on_epoch=True, on_step=False, prog_bar=True)
        return loss

    def training_step(self, batch, _):
        return self._step(batch, "train")

    def validation_step(self, batch, _):
        self._step(batch, "val")

    def configure_optimizers(self):
        opt = torch.optim.Adam(self.parameters(), lr=1e-3)
        sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, patience=3, factor=0.5)
        return {"optimizer": opt, "lr_scheduler": {"scheduler": sched, "monitor": "val_loss"}}


def evaluate(name, path, n_classes=None, n_targets=None, batch_size=64, balanced=False):
    """Load dataset, subsample, train with early stopping, report."""
    task = "regression" if n_targets else "classification"
    ds = DownstreamDataset(path, transform=z_normalize)
    x, y = ds[0]
    n_ch, n_t = x.shape
    full_size = len(ds)

    # Print stats from full dataset before subsampling
    print(f"  Total samples: {full_size:,} | Shape: ({n_ch}, {n_t})")
    if task == "classification":
        counts = ds.get_label_counts()
        desc = ds.label_descriptions or [str(k) for k in sorted(counts.keys())]
        for i, d in enumerate(desc):
            print(f"    {d}: {counts.get(i, 0):,}")
    else:
        print(f"    Targets: {y.shape[-1] if y.dim() > 0 else 1} (regression)")

    # Subsample if too large
    if full_size > MAX_SAMPLES:
        indices = torch.randperm(full_size)[:MAX_SAMPLES].tolist()
        ds = Subset(ds, indices)
        print(f"  Using {MAX_SAMPLES:,} / {full_size:,} samples")

    # Split
    n_val = max(1, len(ds) // 5)
    train_ds, val_ds = random_split(ds, [len(ds) - n_val, n_val])

    # Balanced sampling for imbalanced datasets
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=4)

    n_out = n_targets if n_targets else n_classes
    model = SimpleEEG(n_ch, n_out, task)

    trainer = pl.Trainer(
        max_epochs=MAX_EPOCHS, accelerator="auto", devices=1,
        callbacks=[EarlyStopping(monitor="val_loss", patience=5, mode="min", verbose=False)],
        enable_model_summary=False, enable_checkpointing=False,
        enable_progress_bar=True, logger=False,
    )
    trainer.fit(model, train_loader,
        DataLoader(val_ds, batch_size=batch_size, num_workers=4),
    )

    # Summary
    m = trainer.callback_metrics
    stopped = trainer.current_epoch + 1
    if task == "classification":
        print(f"\n  {name}: val_acc={m.get('val_acc', 0):.3f}  val_loss={m.get('val_loss', 0):.3f}"
              f"  train_acc={m.get('train_acc', 0):.3f}  train_loss={m.get('train_loss', 0):.3f}"
              f"  (epoch {stopped})")
    else:
        print(f"\n  {name}: val_mse={m.get('val_loss', 0):.4f}  train_mse={m.get('train_loss', 0):.4f}"
              f"  (epoch {stopped})")
    print()

## EEGMAT (Mental Arithmetic)
Binary classification: baseline vs arithmetic task. 36 subjects, 19 channels, 5s windows.

In [ ]:
evaluate("EEGMAT", f"{PROCESSED}/eegmat", n_classes=2)

## Mumtaz2016 (Depression Detection)
Binary classification: MDD vs healthy control. 64 subjects, 19 channels, 5s windows.

In [ ]:
evaluate("Mumtaz2016", f"{PROCESSED}/mumtaz2016", n_classes=2)

## BCIC-IV-2a (Motor Imagery)
4-class MI: left hand, right hand, feet, tongue. 9 subjects, 22 channels, 4s windows.

In [ ]:
evaluate("BCIC-IV-2a", f"{PROCESSED}/bcic_iv_2a", n_classes=4)

## ISRUC (Sleep Staging)
5-class sleep staging: W, N1, N2, N3, REM. 100 subjects, 6 channels, 30s epochs.

In [ ]:
evaluate("ISRUC", f"{PROCESSED}/isruc", n_classes=5)

## HMC (Sleep Staging)
5-class sleep staging: W, N1, N2, N3, REM. 151 subjects, 4 channels, 30s epochs.

In [ ]:
evaluate("HMC", f"{PROCESSED}/hmc", n_classes=5, batch_size=128)

## PhysioNet-MI (Motor Imagery)
4-class MI: left fist, right fist, both feet, both fists. 109 subjects, 64 channels, 4s windows.

In [ ]:
evaluate("PhysioNet-MI", f"{PROCESSED}/eegmmidb", n_classes=4)

## CHB-MIT (Seizure Detection)
Binary: seizure vs non-seizure. 23 subjects, 19 channels, 10s windows. Extreme class imbalance.

In [ ]:
evaluate("CHB-MIT", f"{PROCESSED}/chbmit", n_classes=2, batch_size=128, balanced=True)

## SHU-MI (Motor Imagery)
Binary MI: left hand vs right hand. 25 subjects, 32 channels, 4s windows.

In [ ]:
evaluate("SHU-MI", f"{PROCESSED}/shu_mi", n_classes=2)

## MoBI (Gait Prediction)
Multi-target regression: 12 joint angles. 8 subjects, 60 channels, 2s windows.

In [ ]:
evaluate("MoBI", f"{PROCESSED}/mobi", n_targets=12, batch_size=256)